# 05. Diseño y documentación del entrenamiento CNN

TensorFlow/Keras construye explícitamente la CNN. Crear una arquitectura no ajusta sus pesos.

**Dataset principal:** `data/nuevo_dataset/cardio_train.csv`. Este cuaderno no ejecuta entrenamientos. Los resultados históricos se leen de archivos conservados; no se sustituyen por nuevas ejecuciones.

In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/nuevo_dataset/cardio_train.csv").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
from src.experimento import (
    ORIGINAL, RESULTADOS, VARIABLES, NOMBRES, cargar_datos, reglas_calidad,
    cargar_metricas, comparar_ejecuciones, tabla_configuracion, verificar_configuracion,
    construir_cnn, tabla_modelo, dibujar_arquitectura, guardar_figura_nueva,
    importancia_guardada, resumen_dataset, verificar_predicciones, tensorflow_disponible,
)

In [2]:
import importlib.util
if not tensorflow_disponible():
    print("TensorFlow no esta disponible en este kernel. Se verifica la configuracion del modelo guardado; no se construye ni se ejecuta model.summary().")
    tabla = verificar_configuracion()
    display(tabla)
else:
    import tensorflow as tf
    from tensorflow.keras import Sequential
    from tensorflow.keras.layers import Input, Conv1D, GlobalAveragePooling1D, Dense, Dropout
    model = Sequential([
        Input(shape=(11, 1)),
        Conv1D(32, kernel_size=3, padding="same", activation="relu"),
        Conv1D(96, kernel_size=3, padding="same", activation="relu"),
        GlobalAveragePooling1D(),
        Dense(256, activation="relu"),
        Dropout(0.30),
        Dense(1, activation="sigmoid"),
    ])
    model.summary()
    assert model.count_params() == 34529
    assert 30000 <= model.count_params() <= 50000
    display(tabla_modelo(model))
    tabla = tabla_modelo(model)

TensorFlow no esta disponible en este kernel. Se verifica la configuracion del modelo guardado; no se construye ni se ejecuta model.summary().


,capa,tipo,salida,parametros,filtros,kernel,activacion,padding,unidades,dropout
0,conv1d,Conv1D,"(11, 32)",128,32.0,3.0,relu,same,NaN,NaN
1,conv1d_1,Conv1D,"(11, 96)",9312,96.0,3.0,relu,same,NaN,NaN
2,global_average_pooling1d,GlobalAveragePooling1D,"(96,)",0,NaN,NaN,—,—,NaN,NaN
3,dense,Dense,"(256,)",24832,NaN,NaN,relu,—,256.0,NaN
4,dropout,Dropout,"(256,)",0,NaN,NaN,—,—,NaN,0.3
5,dense_1,Dense,"(1,)",257,NaN,NaN,sigmoid,—,1.0,NaN


## Configuración de entrenamiento ya ejecutada
Adam, entropía cruzada binaria, batch size 32, máximo 100 épocas y EarlyStopping con restauración de mejores pesos. La ejecución inicial usó paciencia 15 y terminó tras 44 épocas; la definitiva usó paciencia 10 y terminó tras 66. Los pesos restaurados no tienen por qué ser los de la última época. **No se llama a `model.fit` en esta secuencia.**

In [3]:
display(comparar_ejecuciones())
for nombre in ["historial_nueva_cnn.csv", "historial_entrenamiento.csv"]:
    historial = pd.read_csv(RESULTADOS / "tablas" / nombre)
    print(nombre, "épocas:", len(historial), "mejor val_loss: época", int(np.argmin(historial.val_loss)) + 1)
    display(historial.tail(3))

historial_nueva_cnn.csv épocas: 44 mejor val_loss: época 29
historial_entrenamiento.csv épocas: 66 mejor val_loss: época 56


,Protocolo,Entrenamiento,Validación,Prueba,Épocas,Parámetros,Accuracy,Precision,Recall,F1,ROC-AUC
0,StandardScaler,10500,2250,2250,44,34529,0.712444,0.716141,0.693621,0.704701,0.775738
1,MinMaxScaler,10200,2550,2250,66,34529,0.731111,0.732601,0.718778,0.725624,0.788458


,epoch,accuracy,loss,precision,recall,val_accuracy,val_loss,val_precision,val_recall
41,42,0.739524,0.532980,0.753295,0.704139,0.718667,0.566089,0.721402,0.702606
42,43,0.739048,0.532495,0.752520,0.704139,0.720889,0.565931,0.728990,0.693621
43,44,0.738762,0.531347,0.753306,0.701829,0.716889,0.566548,0.722015,0.695418


,accuracy,loss,val_accuracy,val_loss
63,0.736373,0.540953,0.727843,0.551225
64,0.734510,0.539602,0.728235,0.551100
65,0.734510,0.540457,0.727059,0.551533


El modelo construido arriba tiene pesos recién inicializados y se usa solo para demostrar la arquitectura. El modelo entrenado que respalda las métricas es `resultados/nuevo_experimento/modelo_nueva_cnn.keras`.